<a href="https://colab.research.google.com/github/ShreyIND/ML/blob/main/Cat_dog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile as zf
zip_ref=zf.ZipFile("/content/train.zip",'r')
zip_ref.extractall('/content')
zip_ref.close()
zip_ref

<zipfile.ZipFile [closed]>

In [ ]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import Sequential,layers
from tensorflow.keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization

In [ ]:
train_ds=image_dataset_from_directory(directory='/content/train',image_size=(256,256),batch_size=32,labels='inferred',label_mode='int')

Found 25000 files belonging to 2 classes.


In [ ]:
def process(image,label):
  image=tensorflow.cast(image/255.0,tensorflow.float32)
  return image,label
train_ds=train_ds.map(process)

In [ ]:
from tensorflow.keras import optimizers
def build_cat_dog_model(img_size=256):
    model = Sequential([
        layers.InputLayer(input_shape=(img_size, img_size, 3)),
        layers.Rescaling(1./255),

        # 3. Data Augmentation (Prevents overfitting by slightly altering images)
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),

        # --- FEATURE EXTRACTION BLOCKS (The "Eye") ---
        # Block 1: Detects simple edges/colors
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(), # Stabilizes training
        layers.MaxPooling2D((2, 2)),

        # Block 2: Detects textures (fur, eyes)
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 3: Detects complex shapes (ears, tails)
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 4: Deep abstraction
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # --- CLASSIFICATION HEAD (The "Brain") ---
        layers.Flatten(),

        # Dense Layer with Dropout (Dropout drops 50% neurons to force robustness)
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),

        # Output Layer (1 neuron = Binary Classification)
        layers.Dense(1, activation='sigmoid')
    ])

    # Compile with a safe learning rate
    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizers.Adam(learning_rate=0.001),
        metrics=['accuracy']
    )

    return model

# Build and Summarize
model = build_cat_dog_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 256, 256, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 65536)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 512)            │    33,554,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,945,793 (129.49 MB)

 Trainable params: 33,944,833 (129.49 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
model.fit(train_ds,epochs=10)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 142s 171ms/step - accuracy: 0.6100 - loss: 5.0561
Epoch 2/10
557/782 ━━━━━━━━━━━━━━━━━━━━ 38s 170ms/step - accuracy: 0.6971 - loss: 0.5890